# TP Jour 2 — Modèle d'Exécution Spark, Jointures Multi-Datasets & Couche Silver (Delta Lake)

### Objectifs de la séance
1. Observer la **Lazy Evaluation** et distinguer transformations **Narrow** vs **Wide (Shuffle)** sur **Spark Web UI** (`http://localhost:4040`).
2. Profiler la qualité de vos tables **Bronze** (issues d'un ou **plusieurs datasets Kaggle**), nettoyer les anomalies, typer fortement les colonnes et harmoniser les **clés de jointure inter-sources**.
3. Mettre en œuvre une **jointure distribuée multi-datasets maîtrisée** (`BroadcastHashJoin` via `F.broadcast` pour un référentiel/dataset compact, ou `SortMergeJoin` entre deux grandes sources) **en contrôlant la cardinalité (absence d'explosion N-à-M)**.
4. Sauvegarder votre table analytique enrichie dans **`/lakehouse/silver/`** et expérimenter les garanties **ACID de Delta Lake** (`_delta_log`, **Schema Enforcement**, **`MERGE INTO`**, **Time Travel**).

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .appName("J2_Spark_Delta_Silver")
    .getOrCreate()
)

print("Session Spark active — Spark UI : http://localhost:4040")

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6be2fa60-98a6-4052-a269-45cf89fb0ef7;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found

Session Spark active — Spark UI : http://localhost:4040


## 1. Chargement de vos tables Bronze (Multi-Datasets / Multi-Tables) & Diagnostic Qualité

### Exercice 1.1 — Charger toutes vos tables Bronze et profiler valeurs manquantes / doublons
1. Chargez dans `bronze_dfs` votre table de faits principale ainsi que la (ou les) table(s) secondaire(s) à joindre depuis `/lakehouse/bronze/<nom_table>`.
2. Comptez le nombre de valeurs `NULL` sur vos colonnes clés (en particulier les **clés de jointure**) ainsi que les doublons éventuels.

In [ ]:
BRONZE_TABLES = {
    "main": "/lakehouse/bronze/orders",
    "customers": "/lakehouse/bronze/customers",
    "items": "/lakehouse/bronze/order_items",
    "products": "/lakehouse/bronze/products",
    "payments": "/lakehouse/bronze/payments",
    "reviews": "/lakehouse/bronze/reviews",
}

bronze_dfs = {
    alias: spark.read.format("delta").load(path)
    for alias, path in BRONZE_TABLES.items()
}
for alias, df in bronze_dfs.items():
    print(f"[{alias}] {df.count():,} lignes | colonnes : {df.columns[:10]}")

df_bronze = bronze_dfs["main"]
null_report = df_bronze.select([
    F.sum(F.col(c).isNull().cast("long")).alias(c)
    for c in df_bronze.columns
])
null_report.show(truncate=False)

print("Doublons order_id :", df_bronze.count() - df_bronze.dropDuplicates(["order_id"]).count())

26/11/20 21:06:41 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

[main] 99,441 lignes | colonnes : ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', '_ingested_at', '_source_dataset']


[customers] 99,441 lignes | colonnes : ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', '_ingested_at', '_source_dataset', '_source_file']


[items] 112,650 lignes | colonnes : ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', '_ingested_at', '_source_dataset', '_source_file']


[products] 32,951 lignes | colonnes : ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', '_ingested_at']


[Stage 26:====================================>                   (33 + 4) / 50]

## 2. Harmonisation Inter-Datasets, Nettoyage & Jointure Distribuée (`BroadcastHashJoin` / `SortMergeJoin`)

### Bonnes pratiques de jointure entre plusieurs datasets hétérogènes :
1. **Harmonisation des clés de jointure (*Key Alignment*)** : deux datasets Kaggle d'auteurs différents n'ont presque jamais exactement le même format de clé :
   - *Axe temporel* : ramenez les horodatages à la même granularité et au même fuseau (`F.to_date(...)`, `F.date_trunc("hour", ...)` ou format `yyyy-MM`).
   - *Axe géographique / catégoriel* : normalisez la casse, les espaces et les codes (`F.upper(F.trim(col))`, code ISO pays/région, code postal ou grille `F.round(lat, 2)`).
2. **Prévention des explosions N-à-M** : avant un `LEFT JOIN` de la table principale (`N` lignes) vers un second dataset, assurez-vous que la clé de jointure est **unique côté table secondaire** via une pré-agrégation métier (`groupBy(join_keys).agg(...)`) ou `dropDuplicates(join_keys)` ! Vérifiez systématiquement que le nombre de lignes après `LEFT JOIN` n'explose pas.
3. **Choix de la stratégie physique Spark** :
   - **Table secondaire compacte (< quelques dizaines de Mo)** : forcez **`F.broadcast(df_sec)`** pour obtenir un `BroadcastHashJoin` sans *Shuffle* de la grande table.
   - **Deux grands datasets volumineux** : laissez Spark exécuter un `SortMergeJoin` distribué (optimisé dynamiquement par *Adaptive Query Execution*).

### Exercice 2.1 — Nettoyer, harmoniser vos clés, joindre vos sources et écrire `/lakehouse/silver/<table_silver>`
1. Nettoyez vos tables (`dropDuplicates`, filtrage d'anomalies, typage fort `F.to_timestamp` / `F.to_date` / numériques) et harmonisez la clé de jointure commune.
2. Créez au moins **2 nouvelles colonnes métier dérivées** (dont idéalement un indicateur combinant vos deux sources !) utiles pour les Jours 3 & 4.
3. Réalisez la jointure (avec `F.broadcast(...)` sur une table de dimension/dataset secondaire compact), contrôlez le **taux de rapprochement (*match rate*)** ainsi que l'absence de duplication N-M, et affichez `.explain("simple")`.
4. Écrivez la table enrichie au format **Delta** dans **`/lakehouse/silver/<votre_table>`**.

In [ ]:
SILVER_MAIN_PATH = "/lakehouse/silver/orders_enriched"

orders = bronze_dfs["main"].dropDuplicates(["order_id"])
customers = bronze_dfs["customers"].dropDuplicates(["customer_id"])

# Agrégation des lignes d'articles pour obtenir une seule ligne par commande
items = (
    bronze_dfs["items"]
    .groupBy("order_id")
    .agg(
        F.count("*").alias("item_count"),
        F.sum(F.col("price").cast("double")).alias("items_value"),
        F.sum(F.col("freight_value").cast("double")).alias("freight_value"),
        F.first("product_id", ignorenulls=True).alias("sample_product_id")
    )
)

# Enrichissement de la commande par la catégorie du premier produit disponible
products = bronze_dfs["products"].select(
    "product_id", "product_category_name"
).dropDuplicates(["product_id"])

items_with_category = (
    bronze_dfs["items"]
    .join(products, "product_id", "left")
    .groupBy("order_id")
    .agg(F.first("product_category_name", ignorenulls=True).alias("product_category"))
)

payments = (
    bronze_dfs["payments"]
    .groupBy("order_id")
    .agg(F.sum(F.col("payment_value").cast("double")).alias("payment_value"))
)

reviews = (
    bronze_dfs["reviews"]
    .groupBy("order_id")
    .agg(F.avg(F.col("review_score").cast("double")).alias("review_score"))
)

orders_clean = (
    orders
    .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp"))
    .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date"))
    .withColumn("order_estimated_delivery_date", F.to_timestamp("order_estimated_delivery_date"))
    .filter(F.col("order_id").isNotNull())
)

customers_clean = customers.select(
    "customer_id", "customer_unique_id", "customer_city", "customer_state",
    "customer_zip_code_prefix"
)

df_silver = (
    orders_clean
    .join(F.broadcast(customers_clean), "customer_id", "left")
    .join(items, "order_id", "left")
    .join(items_with_category, "order_id", "left")
    .join(payments, "order_id", "left")
    .join(reviews, "order_id", "left")
    .withColumn(
        "delivery_days",
        F.when(
            F.col("order_delivered_customer_date").isNotNull(),
            (F.unix_timestamp("order_delivered_customer_date") -
             F.unix_timestamp("order_purchase_timestamp")) / 86400.0
        )
    )
    .withColumn(
        "estimated_days",
        (F.unix_timestamp("order_estimated_delivery_date") -
         F.unix_timestamp("order_purchase_timestamp")) / 86400.0
    )
    .withColumn(
        "delay_days",
        F.when(
            F.col("order_delivered_customer_date").isNotNull(),
            (F.unix_timestamp("order_delivered_customer_date") -
             F.unix_timestamp("order_estimated_delivery_date")) / 86400.0
        )
    )
    .withColumn("is_late", (F.col("delay_days") > 0).cast("int"))
    .withColumn("order_value", F.coalesce("payment_value", "items_value"))
    .withColumn("purchase_date", F.to_date("order_purchase_timestamp"))
    .withColumn("purchase_month", F.date_format("order_purchase_timestamp", "yyyy-MM"))
    .filter(F.col("order_id").isNotNull())
)

count_before_join = orders_clean.count()
print("Commandes avant enrichissement :", count_before_join)
print("Commandes après enrichissement :", df_silver.count())
df_silver.explain("simple")

(
    df_silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_MAIN_PATH)
)

silver_count = spark.read.format("delta").load(SILVER_MAIN_PATH).count()
assert silver_count <= count_before_join, "Explosion de lignes détectée après jointure."
print(f"Table Silver écrite : {SILVER_MAIN_PATH} ({silver_count:,} lignes)")

## 3. Garanties Transactionnelles ACID de Delta Lake

### Exercice 3.1 — Explorer `_delta_log` et vérifier le *Schema Enforcement*
1. Lisez et affichez les premières lignes du fichier JSON de transaction `00000000000000000000.json` situé dans `{SILVER_MAIN_PATH}/_delta_log/`.
2. Tentez d'insérer (`mode("append")`) un petit DataFrame artificiel au schéma incompatible dans `SILVER_MAIN_PATH` et interceptez l'exception levée par Delta Lake.

In [ ]:
# 1. Lecture directe du journal transactionnel JSON sur HDFS
spark.read.text(f"{SILVER_MAIN_PATH}/_delta_log/00000000000000000000.json").show(8, truncate=110)

# 2. TODO : Tester le Schema Enforcement avec un DataFrame au mauvais schéma dans un bloc try/except
bad_df = spark.createDataFrame([("test_1", "colonne_intruse")], ["fake_id", "intruder_col"])
try:
    bad_df.write.format("delta").mode("append").save(SILVER_MAIN_PATH)
except Exception as e:
    print("Schema Enforcement validé — écriture invalide bloquée :", str(e).splitlines()[0])

### Exercice 3.2 — Upsert (`MERGE INTO`) et Voyage dans le Temps (`VERSION AS OF`)
1. Prélevez un petit lot de lignes existantes dans votre table Silver (ou simulez un lot correctif + 1 nouvelle ligne) et appliquez un **`MERGE INTO`** via `DeltaTable.forPath(spark, SILVER_MAIN_PATH)` (`whenMatchedUpdate` / `whenNotMatchedInsert`).
2. Affichez l'historique des commits avec `DESCRIBE HISTORY` et interrogez la version initiale de la table via `.option("versionAsOf", 0)` pour démontrer l'auditabilité temporelle !

In [ ]:
target_delta = DeltaTable.forPath(spark, SILVER_MAIN_PATH)

# MERGE de démonstration : réécriture contrôlée d'une commande existante.
merge_source = spark.read.format("delta").load(SILVER_MAIN_PATH).limit(1)
(
    target_delta.alias("target")
    .merge(merge_source.alias("source"), "target.order_id = source.order_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

history_df = spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_MAIN_PATH}`")
history_df.select("version", "timestamp", "operation", "operationMetrics").show(truncate=80)

version_zero = (
    spark.read.format("delta")
    .option("versionAsOf", 0)
    .load(SILVER_MAIN_PATH)
)
current_version = spark.read.format("delta").load(SILVER_MAIN_PATH)
print("Lignes version 0 :", version_zero.count())
print("Lignes version actuelle :", current_version.count())
assert history_df.count() >= 2, "Le MERGE doit créer une nouvelle version Delta."